# Probe-protocol queue, two cards: deltaz-pairlab

6 runs (control, pairlab_delta_z, pairlab_full) on the probe protocol (K=32, 400k training jets, 20 epochs),
the 2026-10-05 control's command plus each run's flags, drained by two T4s
as a queue. T4 with torch 2.11.0+cu128 pinned and asserted. Kernel budget 2.4 h.

In [ ]:
import subprocess, time
T0 = time.time()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip())

In [ ]:
import base64, io, tarfile, pathlib
CODE_B64 = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode="r:gz").extractall("/kaggle/working/repo")
print(sorted(p.name for p in pathlib.Path("/kaggle/working/repo").iterdir()))

In [ ]:
import subprocess, sys

def run(args, cwd="/kaggle/working/repo"):
    """Run a repo module, stream nothing, return (ok, tail-of-output)."""
    r = subprocess.run([sys.executable, "-m"] + args, cwd=cwd,
                       capture_output=True, text=True)
    out = r.stdout[-4000:]
    if r.returncode != 0:
        out += "\n--- stderr ---\n" + r.stderr[-3000:]
    print(out)
    return r.returncode == 0

In [ ]:
import subprocess, sys, time
WANT = "2.11.0+cu128"
name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print("device:", name)
assert "T4" in name, "paired with T4 runs only; got %r -- stopping before any GPU time is spent" % name

def torch_version():
    return subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                          capture_output=True, text=True).stdout.strip()

v = torch_version()
if v != WANT:
    print("torch", v, "- installing", WANT)
    t0 = time.perf_counter()
    # the local label matters: "torch==2.11.0" is satisfied by any 2.11.0
    # build, so on Kaggle's 2.11.0+cu130 image (2026-11-07) pip did nothing
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==" + WANT,
                        "--index-url", "https://download.pytorch.org/whl/cu128"],
                       capture_output=True, text=True)
    print("pip rc", r.returncode, "in %.0f s" % (time.perf_counter() - t0))
    if r.returncode:
        print(r.stderr[-2000:])
        raise SystemExit("could not install torch " + WANT)
    v = torch_version()
assert v == WANT, "torch %s, expected %s" % (v, WANT)
print("torch", v, "on", name)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchdiffeq"], check=True)

In [ ]:
import os, subprocess, sys

def launch(args, gpu, log):
    """Start a repo module on one card; its output goes to `log`."""
    # unbuffered, with faulthandler: a killed or crashed child still leaves
    # its last lines and a traceback in `log`
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
               PYTHONFAULTHANDLER="1")
    return subprocess.Popen([sys.executable, "-m"] + args, cwd="/kaggle/working/repo",
                            env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)

def tail(log, n=3000):
    return open(log).read()[-n:]

cards = [l for l in subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                   capture_output=True, text=True).stdout.splitlines() if l.strip()]
NGPU = len(cards)
print("cards:", cards)

In [ ]:
import glob, pathlib
cands = glob.glob("/kaggle/input/**/top_tagging_train.npz", recursive=True)
assert cands, "attach the K=128 data-prep kernel output"
DATA = str(pathlib.Path(cands[0]).parent)
TAG, JOBS, BUDGET = 'deltaz-pairlab', [['pairlab_delta_z', ['--pair-lab', '--pair-lab-set', 'delta_z'], 0], ['pairlab_delta_z', ['--pair-lab', '--pair-lab-set', 'delta_z'], 1], ['pairlab_delta_z', ['--pair-lab', '--pair-lab-set', 'delta_z'], 2], ['pairlab_delta_z', ['--pair-lab', '--pair-lab-set', 'delta_z'], 3], ['pairlab_full', ['--pair-lab'], 4], ['control', [], 4]], 2.4 * 3600
OUT = "/kaggle/working/results_probe_queue"
CKPT = "/kaggle/working/checkpoints"
LOGS = "/kaggle/working/logs/" + TAG
RUN_TIME = 0.55 * 3600       # one probe run end to end, two cards busy (0.49-0.60 h for pair_lab)
BASE = ['--cache-dir', 'DATA', '--representation', 'constituents', '--canonical-splits', '--normalize', 'global', '--max-train-samples', '400000', '--epochs', '20', '--device', 'cuda', '--dtype', 'float32', '--batch-size', '256', '--n-constituents', '32', '--eval-chunk-size', '2048', '--models', 'so3c_message_set', '--rounds', '3', '--beams', '--channels', '8', '--compile']
BASE[BASE.index("DATA")] = DATA
for d in (OUT, LOGS):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
pathlib.Path(OUT, "bundle_sha256.txt").write_text('08613db47d6c' + chr(10))
print("data:", DATA, "| bundle", '08613db47d6c', "|", len(JOBS), "jobs")

In [ ]:
# On-card guard: the forward pass must be the published function on this
# GPU too, in float64. "1 passed" is required; a skip means no CUDA.
import subprocess, sys
g = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                    "tests/test_so3c_models.py", "-k", "golden_cuda"],
                   cwd="/kaggle/working/repo", capture_output=True, text=True)
print(g.stdout[-1500:], g.stderr[-1500:])
assert "1 passed" in g.stdout, "golden guard did not pass on the card"

In [ ]:
# Smoke on both cards at once with the first job's flags: one epoch on 20k
# jets per card. Each card must write a result to be given jobs.
import glob
smoke = []
for i in range(NGPU):
    d = "/kaggle/working/smoke/%s/card%d" % (TAG, i)
    smoke.append((d, launch(["benchmarks.run_top_tagging"] + BASE + JOBS[0][1] +
                            ["--epochs", "1", "--max-train-samples", "20000",
                             "--seed", "0", "--results-dir", d],
                            i, LOGS + "/smoke_card%d.log" % i)))
codes = [p.wait() for _, p in smoke]
CARDS = [i for i, ((d, _), rc) in enumerate(zip(smoke, codes)) if glob.glob(d + "/*.json")]
for i, rc in enumerate(codes):
    print("card %d: exit %d, result %s" % (i, rc, i in CARDS))
    if i not in CARDS:
        print(tail(LOGS + "/smoke_card%d.log" % i))
assert CARDS, "the smoke run failed on every card"
print("cards in use:", CARDS)

In [ ]:
# The queue: each free card takes the next job; no job starts once less than
# one run's time is left of the budget.
import glob, time
todo = list(JOBS)
running = {}                     # card -> (arm, seed, process)
BAD, SKIPPED = [], []
while todo or running:
    for c in CARDS:
        if c in running:
            arm, s, p = running[c]
            if p.poll() is None and time.time() - T0 > BUDGET + 0.1 * 3600:
                p.terminate(); p.wait()          # hung past the budget: its checkpoint is kept
                print("card %d: %s seed %d terminated past the budget" % (c, arm, s), flush=True)
            if p.poll() is None:
                continue
            ok = p.returncode == 0 and glob.glob(OUT + "/%s/seed%d/*.json" % (arm, s))
            print("%7.2f h  card %d: %s seed %d %s" % ((time.time() - T0) / 3600, c, arm, s,
                                                       "done" if ok else "FAILED, exit %d" % p.returncode), flush=True)
            if not ok:
                BAD.append((arm, s)); print(tail(LOGS + "/%s_seed%d.log" % (arm, s), 1500))
            del running[c]
        if todo and c not in running:
            if time.time() - T0 + RUN_TIME > BUDGET:
                SKIPPED += [(a, s) for a, _, s in todo]; todo = []
                print("budget: the remaining jobs are skipped:", SKIPPED); continue
            arm, flags, s = todo.pop(0)
            cap = int(min(RUN_TIME * 1.5, BUDGET - (time.time() - T0) - 600))
            running[c] = (arm, s, launch(["benchmarks.run_top_tagging"] + BASE + flags +
                                         ["--seed", str(s), "--results-dir", OUT + "/%s/seed%d" % (arm, s),
                                          "--ckpt-dir", CKPT + "/%s_seed%d" % (arm, s), "--resume",
                                          "--max-seconds", str(cap)],
                                         c, LOGS + "/%s_seed%d.log" % (arm, s)))
            print("%7.2f h  card %d: start %s seed %d" % ((time.time() - T0) / 3600, c, arm, s), flush=True)
    time.sleep(20)
print("failed:", BAD or "none", "| skipped:", SKIPPED or "none")
print("kernel time %.2f h" % ((time.time() - T0) / 3600))

In [ ]:
# Each subset arm this queue runs, unpaired against the pooled earlier probe
# controls and full --pair-lab runs (REFS) together with this kernel's
# anchors, with the reading rule fixed before the push:
#   f = (arm - control) / (full - control), the fraction of the gain recovered;
#   "carries the gain" if f >= 0.7 and the arm beats the control at p < 0.05,
#   "carries none" if f <= 0.3 and the arm is below the full set at p < 0.05,
#   otherwise unresolved.
import json, glob, statistics as st, math
from scipy import stats
REFS = {'control': [(0.9811491732570162, 804.4382470119522), (0.9811092523841787, 824.138775510204), (0.981106801276, 824.138775510204), (0.9809738936815733, 804.4382470119522), (0.9810829106236938, 761.9396226415093), (0.9810854817813994, 834.3553719008264), (0.9809025757586377, 779.5907335907336)], 'pairlab': [(0.9814375514952488, 817.4655870445345), (0.9816360003720188, 837.8174273858921), (0.9817201289163335, 834.3553719008264), (0.9812255429282452, 770.6641221374045), (0.9815074466407478, 848.3781512605043), (0.9815797889732409, 834.3553719008264), (0.9815499801745822, 855.5677966101695)], 'frame': [(0.9815127770627512, 855.5677966101695), (0.9815029301341944, 909.5225225225225), (0.9815711918120904, 804.4382470119522)], 'mass': [(0.9811344809692828, 930.4792626728109), (0.9811386985222482, 773.6168582375478), (0.9813058491260185, 804.4382470119522)]}
def load(arm):
    rs = [json.load(open(f)) for f in sorted(glob.glob(OUT + "/%s/seed*/*.json" % arm))]
    for r in rs:
        t = r["test_metrics"]; env = r["env"]
        assert env["torch"] == "2.11.0+cu128" and "T4" in env.get("gpu", ""), env
        print("%-14s seed %d: AUC %.5f  rej@0.3 %.1f  %d epochs  %.2f h  kwargs %s"
              % (arm, r["seed"], t["test_auc"], t["bg_rej_30"], r["epochs_run"], r["walltime_sec"] / 3600, r["so3c_kwargs"]))
    return [r["test_metrics"]["test_auc"] for r in rs if r["epochs_run"] == 20]
def welch(a, b):
    va, vb = st.variance(a) / len(a), st.variance(b) / len(b)
    t = (st.mean(a) - st.mean(b)) / math.sqrt(va + vb)
    df = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1))
    return st.mean(a) - st.mean(b), t, df, 2 * stats.t.sf(abs(t), df)
new_full, new_ctrl = load("pairlab_full"), load("control")
old_full = [r[0] for r in REFS["pairlab"]]; old_ctrl = [r[0] for r in REFS["control"]]
for name, new, old in (("full pair_lab", new_full, old_full), ("control", new_ctrl, old_ctrl)):
    if new:
        print("anchor %s here %s against the earlier runs' %.5f +- %.5f: offset %+.5f"
              % (name, " ".join("%.5f" % x for x in new), st.mean(old), st.stdev(old), st.mean(new) - st.mean(old)))
full, ctrl = old_full + new_full, old_ctrl + new_ctrl
print("references: control %d runs %.5f +- %.5f | full pair_lab %d runs %.5f +- %.5f"
      % (len(ctrl), st.mean(ctrl), st.stdev(ctrl), len(full), st.mean(full), st.stdev(full)))
ARMS = [a for a in dict.fromkeys(j[0] for j in JOBS) if a not in ("pairlab_full", "control")]
for arm in ARMS:
    a = load(arm)
    if len(a) < 2:
        print("  %s: %d complete runs, no reading" % (arm, len(a))); continue
    d, t, df, pc = welch(a, ctrl); d2, t2, df2, pf = welch(a, full)
    A, C, F = st.mean(a), st.mean(ctrl), st.mean(full); D = F - C
    vA, vC, vF = st.variance(a) / len(a), st.variance(ctrl) / len(ctrl), st.variance(full) / len(full)
    f = (A - C) / D
    se = math.sqrt(vA / D ** 2 + vF * ((A - C) / D ** 2) ** 2 + vC * ((A - F) / D ** 2) ** 2)
    if f >= 0.7 and d > 0 and pc < 0.05:
        verdict = "carries the gain"
    elif f <= 0.3 and d2 < 0 and pf < 0.05:
        verdict = "carries none of the gain"
    else:
        verdict = "unresolved"
    print("  %s: mean %.5f +- %.5f | vs control %+.5f (t %.1f, df %.1f, p %.3f) | vs full %+.5f (t %.1f, df %.1f, p %.3f) | fraction of the gain %.2f +- %.2f -> %s"
          % (arm, A, st.stdev(a), d, t, df, pc, d2, t2, df2, pf, f, se, verdict))
    # context, no verdict: this kernel's anchors alone, and the 10-18 arms
    if new_full and new_ctrl and abs(st.mean(new_full) - st.mean(new_ctrl)) > 1e-9:
        print("    against this kernel's anchors alone (one run each): fraction %.2f"
              % ((A - st.mean(new_ctrl)) / (st.mean(new_full) - st.mean(new_ctrl))))
    for name in ("frame", "mass"):
        if name in REFS and arm != "pairlab_" + name:
            other = [r[0] for r in REFS[name]]
            d5, t5, df5, p5 = welch(a, other)
            print("    against the 2026-10-18 %s arm (%d runs, %.5f): %+.5f (t %.1f, df %.1f, p %.3f)"
                  % (name, len(other), st.mean(other), d5, t5, df5, p5))
assert not BAD, "jobs failed: %s" % BAD